# 📘 The AI Engineer's LLM Workbook

**14 Chapters · 14 Google Colab Notebooks · Beginner to Production**

---

*© 2026 JAWNVION LLC — www.jawnvion.com — peter@jawnvion.com*

*Licensed for individual use. Do not redistribute.*

---

## What's Inside

| # | Chapter |
|---|---------|
| 01 | AI Fundamentals & Problem Framing |
| 02 | Data Science Toolkit (NumPy, Pandas, Matplotlib) |
| 03 | Neural Networks from Scratch |
| 04 | Transformers Architecture Deep Dive |
| 05 | HuggingFace & Pre-Trained Models |
| 06 | QLoRA Fine-Tuning |
| 07 | DPO Alignment Training |
| 08 | Retrieval-Augmented Generation (RAG) |
| 09 | Model Evaluation & Benchmarking |
| 10 | FastAPI Deployment |
| 11 | Monitoring & Observability |
| 12 | Security for AI Systems |
| 13 | Cost Optimization & Quantization |
| 14 | Capstone: End-to-End LLM Project |

---

> **How to use:** Click **Runtime → Run All** in Google Colab, or run cells one at a time.
> Each chapter builds on the last — complete them in order for best results.

---


# Chapter 4: Transformers & Attention Mechanisms
**JAWNVION LLC — AI Training Workbook**

Every modern LLM — GPT-4, TinyLlama, Llama 3 — is a Transformer.
Understanding the Transformer architecture means you can read any LLM paper,
debug any fine-tuning run, and understand exactly what your model is doing.

**What you'll learn:**
- Why RNNs failed and what Transformers solved
- Scaled dot-product attention — the core operation, step by step
- Multi-head attention — why multiple "perspectives" matter
- Positional encoding — how the model knows word order
- The full Transformer block (attention + FFN + residual + LayerNorm)
- How TinyLlama's architecture maps to these concepts

**No GPU required** — we implement everything in NumPy for clarity.

In [ ]:
# — Cell 1: Why Transformers? The Problem with RNNs ——
print("THE PROBLEM TRANSFORMERS SOLVED")
print("=" * 55)
print()
problems = [
    ("Sequential processing",
     "RNNs process tokens one at a time — can't be parallelised.\n"
     "     Transformers process ALL tokens simultaneously."),
    ("Long-range dependencies",
     "RNNs 'forget' context from 100+ tokens ago.\n"
     "     Attention has DIRECT connections between any two tokens."),
    ("Gradient vanishing",
     "Gradients shrink exponentially through long RNN chains.\n"
     "     Transformers use residual connections — gradients flow freely."),
    ("Fixed context window",
     "RNNs have fixed hidden state size.\n"
     "     Transformers scale context by extending the attention matrix."),
]
for title, desc in problems:
    print(f"  ✗ {title}")
    print(f"     {desc}")
    print()

print("The 2017 paper 'Attention Is All You Need' (Vaswani et al.) introduced")
print("the Transformer architecture. It is still the foundation of every LLM today.")
print()
print("TinyLlama is a DECODER-ONLY Transformer — the same design as GPT.")
print("It has 22 layers, each containing self-attention + a feed-forward network.")

In [ ]:
# — Cell 2: Scaled Dot-Product Attention — Step by Step
import numpy as np

# Attention answers: "for each token, which other tokens should I attend to?"
# Input: a sequence of token embeddings (here: 5 tokens, dim=8)

np.random.seed(42)

SEQ_LEN  = 5    # number of tokens
D_MODEL  = 8    # embedding dimension
D_K      = 4    # key/query dimension

tokens = ["The", "quick", "brown", "fox", "jumps"]

# 1. Create Q, K, V projection matrices (learned during training)
W_Q = np.random.randn(D_MODEL, D_K)
W_K = np.random.randn(D_MODEL, D_K)
W_V = np.random.randn(D_MODEL, D_K)

# 2. Input embeddings (normally from an embedding lookup table)
X = np.random.randn(SEQ_LEN, D_MODEL)  # shape: (5, 8)

# 3. Project to Q, K, V
Q = X @ W_Q    # queries:  (5, 4)
K = X @ W_K    # keys:     (5, 4)
V = X @ W_V    # values:   (5, 4)

# 4. Attention scores = Q @ K^T / sqrt(d_k)
scores = Q @ K.T / np.sqrt(D_K)   # (5, 5)

# 5. Apply causal mask (decoder-only: can't see future tokens)
mask = np.triu(np.ones((SEQ_LEN, SEQ_LEN)), k=1) * -1e9
scores_masked = scores + mask

# 6. Softmax → attention weights (rows sum to 1)
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

attn_weights = softmax(scores_masked)  # (5, 5)

# 7. Weighted sum of values
output = attn_weights @ V   # (5, 4)

print("SCALED DOT-PRODUCT ATTENTION")
print("=" * 50)
print(f"  Input shape X      : {X.shape}")
print(f"  Query  Q = X @ W_Q : {Q.shape}")
print(f"  Key    K = X @ W_K : {K.shape}")
print(f"  Value  V = X @ W_V : {V.shape}")
print(f"  Scores Q @ K.T / √{D_K}: {scores.shape}")
print(f"  Attention weights   : {attn_weights.shape}")
print(f"  Output attn @ V     : {output.shape}")
print()
print("Attention weights (rows=queries, cols=keys):")
print("  Tokens:", tokens)
header = f"{'':>8}" + "".join(f"{t:>8}" for t in tokens)
print(f"  {header}")
for i, row in enumerate(attn_weights):
    vals = "".join(f"{v:>8.3f}" for v in row)
    print(f"  {tokens[i]:>8}{vals}")
print()
print("Note: lower-triangle only (causal mask prevents attending to future tokens).")
print("This is what makes decoder-only LLMs generate text left-to-right.")

In [ ]:
# — Cell 3: Visualise Attention Weights ———————————————
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: heatmap of attention weights
im = axes[0].imshow(attn_weights, cmap='Blues', vmin=0, vmax=1)
axes[0].set_xticks(range(SEQ_LEN))
axes[0].set_yticks(range(SEQ_LEN))
axes[0].set_xticklabels(tokens, fontsize=11)
axes[0].set_yticklabels(tokens, fontsize=11)
axes[0].set_title('Causal Self-Attention Weights\n(row = query token, col = key token)',
                   fontsize=11, fontweight='bold')
axes[0].set_xlabel('Key (attended to)')
axes[0].set_ylabel('Query (attending from)')
plt.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)
for i in range(SEQ_LEN):
    for j in range(SEQ_LEN):
        if attn_weights[i, j] > 0.01:
            axes[0].text(j, i, f'{attn_weights[i,j]:.2f}',
                        ha='center', va='center', fontsize=8,
                        color='white' if attn_weights[i,j] > 0.5 else 'black')

# Right: raw scores vs masked scores
x_pos = np.arange(SEQ_LEN)
width = 0.35
axes[1].bar(x_pos - width/2, scores[2],  width, label='Raw scores (fox→*)',  color='#3498db', alpha=0.8)
axes[1].bar(x_pos + width/2, attn_weights[2], width, label='Softmax weights', color='#e74c3c', alpha=0.8)
axes[1].axvline(2.5, color='black', linewidth=1.5, linestyle='--', label='Causal boundary')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(tokens, fontsize=10)
axes[1].set_title('"brown" attending to context\n(scores → softmax → weights)',
                   fontsize=11, fontweight='bold')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/content/ch4_attention.png', dpi=120, bbox_inches='tight')
plt.show()
print("✓  Attention visualisation saved to /content/ch4_attention.png")

In [ ]:
# — Cell 4: Multi-Head Attention — Multiple Perspectives ——
import numpy as np

# Single-head attention asks ONE question: "which tokens are relevant?"
# Multi-head attention asks N_HEADS DIFFERENT questions simultaneously:
#   Head 1: "which tokens are syntactically related?"
#   Head 2: "which tokens share semantic meaning?"
#   Head 3: "which tokens are entities?"
#   etc.

def single_head_attention(X, W_Q, W_K, W_V, causal=True):
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V
    d_k    = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    if causal:
        n = scores.shape[0]
        mask = np.triu(np.ones((n, n)), k=1) * -1e9
        scores = scores + mask
    def softmax(x):
        x = x - x.max(axis=-1, keepdims=True)
        e = np.exp(x)
        return e / e.sum(axis=-1, keepdims=True)
    weights = softmax(scores)
    return weights @ V, weights

def multi_head_attention(X, n_heads=4, d_model=8):
    """Concatenate n_heads independent attention operations, then project."""
    assert d_model % n_heads == 0
    d_k = d_model // n_heads
    np.random.seed(7)

    head_outputs = []
    head_weights = []
    for h in range(n_heads):
        W_Q = np.random.randn(d_model, d_k)
        W_K = np.random.randn(d_model, d_k)
        W_V = np.random.randn(d_model, d_k)
        out, w = single_head_attention(X, W_Q, W_K, W_V, causal=True)
        head_outputs.append(out)
        head_weights.append(w)

    # Concatenate all heads: (seq, n_heads * d_k) = (seq, d_model)
    concat = np.concatenate(head_outputs, axis=-1)
    # Final linear projection
    W_O = np.random.randn(d_model, d_model)
    output = concat @ W_O    # (seq, d_model)
    return output, head_weights

N_HEADS = 4
D_MODEL = 8

output, head_weights = multi_head_attention(X, n_heads=N_HEADS, d_model=D_MODEL)

print("MULTI-HEAD ATTENTION")
print("=" * 50)
print(f"  Input X shape   : {X.shape}")
print(f"  n_heads         : {N_HEADS}")
print(f"  d_k per head    : {D_MODEL // N_HEADS}")
print(f"  Each head output: (5, {D_MODEL // N_HEADS})")
print(f"  After concat    : (5, {D_MODEL})")
print(f"  After projection: {output.shape}")
print()
print("TinyLlama uses 32 attention heads across 22 layers.")
print("Each head specialises in different linguistic/semantic patterns.")
print()
print("Attention weight comparison across 4 heads (for token 'brown'):")
print(f"  {'Token':<8}", end="")
for h in range(N_HEADS):
    print(f"  Head {h+1}", end="")
print()
for j, tok in enumerate(tokens):
    print(f"  {tok:<8}", end="")
    for h in range(N_HEADS):
        w = head_weights[h][2, j]   # 'brown' query (index 2)
        print(f"  {w:>6.3f}", end="")
    print()

In [ ]:
# — Cell 5: Positional Encoding — Teaching Order to Attention ——
import numpy as np
import matplotlib.pyplot as plt

# Attention is PERMUTATION-INVARIANT — "cat sat mat" and "mat sat cat"
# would produce the same attention weights without positional information.
# Positional encoding injects position awareness into the embeddings.

# Original Transformer: sinusoidal positional encoding
def sinusoidal_pe(seq_len, d_model):
    pe = np.zeros((seq_len, d_model))
    pos = np.arange(seq_len)[:, np.newaxis]       # (seq, 1)
    div = np.exp(np.arange(0, d_model, 2) *
                 -(np.log(10000.0) / d_model))    # (d_model/2,)
    pe[:, 0::2] = np.sin(pos * div)
    pe[:, 1::2] = np.cos(pos * div)
    return pe

SEQ_LEN  = 20
D_MODEL  = 64
pe = sinusoidal_pe(SEQ_LEN, D_MODEL)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: heatmap of PE matrix
im = axes[0].imshow(pe.T, aspect='auto', cmap='RdYlBu', vmin=-1, vmax=1)
axes[0].set_xlabel('Position in sequence')
axes[0].set_ylabel('Embedding dimension')
axes[0].set_title('Sinusoidal Positional Encoding\n(original Transformer)',
                   fontsize=11, fontweight='bold')
plt.colorbar(im, ax=axes[0])

# Right: a few dimensions over positions
for dim in [0, 4, 16, 32]:
    axes[1].plot(pe[:, dim], linewidth=2, label=f'dim {dim}')
axes[1].set_xlabel('Position in sequence')
axes[1].set_ylabel('Encoding value')
axes[1].set_title('Sinusoidal PE — Individual Dimensions\n(each dim = unique position fingerprint)',
                   fontsize=11, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/ch4_positional_encoding.png', dpi=120, bbox_inches='tight')
plt.show()
print("✓  Plot saved to /content/ch4_positional_encoding.png")
print()
print("Note: TinyLlama uses ROTARY POSITIONAL EMBEDDINGS (RoPE) instead of sinusoidal.")
print("RoPE encodes position relative to each token pair — better for long contexts.")
print("When you load TinyLlama in Ch5 and check its config, you'll see 'rope_theta'.")

In [ ]:
# — Cell 6: The Full Transformer Block ——————————————————
import numpy as np

# One Transformer block (layer) consists of:
#   1. Multi-Head Self-Attention  (with residual connection + LayerNorm)
#   2. Feed-Forward Network       (with residual connection + LayerNorm)

def layer_norm(x, eps=1e-6):
    """Normalise across the embedding dimension."""
    mean = x.mean(axis=-1, keepdims=True)
    std  = x.std(axis=-1, keepdims=True)
    return (x - mean) / (std + eps)

def relu(x):
    return np.maximum(0, x)

def feed_forward(x, d_model, d_ff):
    """Two-layer FFN with ReLU. TinyLlama uses SiLU + a gate projection instead."""
    np.random.seed(1)
    W1 = np.random.randn(d_model, d_ff)  * 0.02
    b1 = np.zeros(d_ff)
    W2 = np.random.randn(d_ff,   d_model) * 0.02
    b2 = np.zeros(d_model)
    h  = relu(x @ W1 + b1)
    return h @ W2 + b2

D_MODEL = 8
D_FF    = 32   # FFN hidden dim is typically 4× d_model

# Simulate input (seq tokens after embedding + positional encoding)
np.random.seed(0)
x_input = np.random.randn(SEQ_LEN, D_MODEL)

# ── Sub-layer 1: Multi-Head Self-Attention ───────────────
attn_out, _ = multi_head_attention(x_input, n_heads=4, d_model=D_MODEL)
x_after_attn = layer_norm(x_input + attn_out)   # residual + norm

# ── Sub-layer 2: Feed-Forward Network ───────────────────
ffn_out = feed_forward(x_after_attn, D_MODEL, D_FF)
x_out   = layer_norm(x_after_attn + ffn_out)    # residual + norm

print("TRANSFORMER BLOCK — DATA FLOW")
print("=" * 55)
print(f"  Input                       : {x_input.shape}")
print()
print(f"  [Sub-layer 1: Self-Attention]")
print(f"  Multi-head attention output : {attn_out.shape}")
print(f"  Residual: x + attn_out      : {(x_input + attn_out).shape}")
print(f"  After LayerNorm             : {x_after_attn.shape}")
print()
print(f"  [Sub-layer 2: Feed-Forward]")
print(f"  FFN output                  : {ffn_out.shape}")
print(f"  Residual: x + ffn_out       : {(x_after_attn + ffn_out).shape}")
print(f"  After LayerNorm (= block out): {x_out.shape}")
print()
print("TinyLlama stacks 22 of these blocks.")
print("Each block refines the representation — from tokens → meaning.")
print()
print("The RESIDUAL CONNECTIONS (x + sublayer(x)) are critical:")
print("  → Gradients flow directly back to early layers (no vanishing)")
print("  → Model can 'skip' a layer if it adds no useful transformation")

In [ ]:
# — Cell 7: TinyLlama Architecture — By the Numbers ————
print("TINYLLAMA 1.1B — ARCHITECTURE BREAKDOWN")
print("=" * 60)
print()

config = {
    "Model family":          "Llama 2 architecture (decoder-only Transformer)",
    "Total parameters":      "1,100,048,384  (~1.1 billion)",
    "Embedding dim (d_model)":"2,048",
    "Number of layers":      "22  (Transformer blocks)",
    "Attention heads":       "32  (multi-head self-attention)",
    "KV heads":              "4   (grouped-query attention — GQA)",
    "FFN hidden dim":        "5,632  (SwiGLU gated FFN)",
    "Activation function":   "SiLU (Swish) — smooth, gated",
    "Positional encoding":   "RoPE (Rotary Position Embedding)",
    "Context length":        "2,048 tokens (max sequence length)",
    "Vocabulary size":       "32,000 tokens (SentencePiece BPE)",
    "Normalisation":         "RMSNorm (faster than LayerNorm)",
    "Tied embeddings":       "Yes — input & output embedding share weights",
}

for key, val in config.items():
    print(f"  {key:<28} {val}")

print()
print("Parameter count estimation:")
emb    = 32000 * 2048
attn   = 22 * (2048 * 2048 * 4)   # Q,K,V,O per layer (simplified)
ffn    = 22 * (2048 * 5632 * 3)   # gate,up,down per layer
total  = emb + attn + ffn
print(f"  Embedding table  : {emb:>15,}")
print(f"  Attention layers : {attn:>15,}")
print(f"  FFN layers       : {ffn:>15,}")
print(f"  Estimated total  : {total:>15,}")
print(f"  Actual total     : {'1,100,048,384':>15}")
print()
print("You'll load this model for real in Chapter 5.")

In [ ]:
# — Cell 8: Chapter 4 Vocabulary ——————————————————————
glossary = {
    "Attention":          "Mechanism that lets each token 'look at' all other tokens.",
    "Query / Key / Value":"Q=what I'm looking for, K=what I offer, V=what I contain.",
    "Scaled dot-product": "Attention score = (Q @ K.T) / sqrt(d_k), then softmax.",
    "Causal mask":        "Prevents tokens from attending to future positions (decoder).",
    "Multi-head":         "Run N independent attention operations, concat, project.",
    "Residual connection":"Add input to sublayer output — prevents vanishing gradient.",
    "LayerNorm / RMSNorm":"Normalise each token's representation — stabilises training.",
    "FFN":                "Feed-Forward Network — two linear layers after attention.",
    "RoPE":               "Rotary Position Embedding — encodes relative position.",
    "GQA":                "Grouped-Query Attention — fewer KV heads, less VRAM.",
    "SwiGLU / SiLU":      "Gated activation function — better than ReLU for LLMs.",
    "d_model":            "Embedding dimension — the 'width' of the model.",
    "d_ff":               "FFN hidden size — typically 2.75× or 4× d_model.",
    "Context length":     "Maximum number of tokens the model can process at once.",
}
print("CHAPTER 4 — CORE VOCABULARY")
print("=" * 65)
for term, defn in glossary.items():
    print(f"  {term:<22}  {defn}")
print()
print("Every term above appears in TinyLlama's config.json.")
print("In Ch5 you'll read that config. In Ch6 you'll fine-tune the model.")

## ✓ Chapter 4 Complete

You now understand the Transformer architecture at the level of the original paper —
attention scores, masking, multi-head projection, residual connections, and how
TinyLlama's specific design choices (RoPE, GQA, SwiGLU) improve on the baseline.

**Next:** Chapter 5 — Hugging Face Ecosystem
You'll load TinyLlama, explore its tokenizer, run your first inference, and
prepare the environment for fine-tuning in Chapter 6.

---
*JAWNVION LLC AI Training Workbook · peter@jawnvion.com*